In [ ]:
import os

from inlist import target, PIPELINE_OPTIONS_OVERRIDE
from run_context import apply_context_globals
from atmos_lbl_pipeline import step_log
from abund_lbl_pipeline import (
    load_star_and_atmosphere_context,
    plot_all_element_gaussian_fits,
    run_nonfe_line_review,
    apply_lineqa_for_abundance,
    prepare_abundance_engine,
    compute_lbl_differential_and_export,
    run_hfs_blends_and_update_outputs,
)

ctx = apply_context_globals(
    globals(),
    target=target,
    pipeline_options_override=PIPELINE_OPTIONS_OVERRIDE,
    prepare_io=True,
)

The atmosphere dump from step 3a supplies fixed Teff, logg, model metallicity, and microturbulence. Compute EW abundances with ispec.determine_abundances(code="moog") and subtract the same-instrument solar abundance line by line. Use the fitted linemasks carrying atomic data and QA decisions.

In [ ]:
step_log("1", "Load context inputs for element abundance workflow.")
_ctx4a = load_star_and_atmosphere_context(
    ispec=ispec,
    target=target,
    output_folder=output_folder,
    spectrum_norm_path=spectrum_norm_path,
    output_atmos_params_dumpfile_path=output_atmos_params_dumpfile_path,
    initial_teff=initial_teff,
    initial_logg=initial_logg,
    initial_MH=initial_MH,
    ispec_dir=ispec_dir,
    pipeline_options=PIPELINE_OPTIONS,
)

star_spectrum = _ctx4a["star_spectrum"]
linemask_output_folder = _ctx4a["linemask_output_folder"]
linemasks = _ctx4a["linemasks"]
status_q2 = _ctx4a["status_q2"]
teff = _ctx4a["teff"]
logg = _ctx4a["logg"]
mh = _ctx4a["mh"]
alpha = _ctx4a["alpha"]
vmic = _ctx4a["vmic"]


In [ ]:
step_log("2", "Run Gaussian diagnostics for all non-Fe elements.")
RUN_FIND_LINEMASKS = PIPELINE_OPTIONS.get("run_find_linemasks", True)
USE_EXTERNAL_EW = PIPELINE_OPTIONS.get("input_has_external_ews", False)
RUN_PART1_SPECTRUM = RUN_FIND_LINEMASKS and (not USE_EXTERNAL_EW)
PLOT_All_Elements = True

_plot_res = plot_all_element_gaussian_fits(
    star_spectrum=star_spectrum,
    linemasks=linemasks,
    output_folder=output_folder,
    run_part1_spectrum=RUN_PART1_SPECTRUM,
    plot_all_elements=PLOT_All_Elements,
    progress_every=10,
)
fig_ele_dir = _plot_res["base_output_dir"]
step_log("2", f"Element Gaussian diagnostics ready: dir={fig_ele_dir}")

In [ ]:
step_log("3", "Run interactive line review for non-Fe Gaussian plots.")
run_nonfe_line_review(
    ispec=ispec, star_spectrum=star_spectrum, linemasks=linemasks,
    output_folder=output_folder,
    target=target,
    run_part1_spectrum=RUN_PART1_SPECTRUM,
    plot_all_elements=PLOT_All_Elements,
    pipeline_options=PIPELINE_OPTIONS,
)

In [ ]:
step_log("4", "Apply LineQA updates to linemasks for abundance computation.")
linemasks = apply_lineqa_for_abundance(
    ispec=ispec,
    linemasks=linemasks,
    output_folder=output_folder,
    linemask_output_folder=linemask_output_folder,
    target=target,
    pipeline_options=PIPELINE_OPTIONS,
)

## Element abundances

Use the MARCS.GES atmosphere grid and Grevesse.2007/stdatom.dat abundance scale. Atmospheric metallicity comes from q2; the final Fe abundance is remeasured with the same backend as the other elements.

In [ ]:
step_log("5", "Prepare abundance engine (model atmosphere + solar references).")
linemasks = linemasks[linemasks["wave_nm"] > 0]
linemasks = linemasks[linemasks["ew"] > 0]

_engine = prepare_abundance_engine(
    ispec=ispec,
    ispec_dir=ispec_dir,
    teff=teff,
    logg=logg,
    mh=mh,
    alpha=alpha,
    code="moog",
)
code = _engine["code"]
model = _engine["model"]
solar_abundances_file = _engine["solar_abundances_file"]
solar_abundances = _engine["solar_abundances"]
atmosphere_layers = _engine["atmosphere_layers"]
step_log("5", f"Abundance engine ready: model={model}")

## Line-by-line differential abundances

In [ ]:

step_log("6", "Compute LBL differential abundances and export outputs.")
_lbl = compute_lbl_differential_and_export(
    ispec=ispec,
    linemasks=linemasks,
    target=target,
    ispec_dir=ispec_dir,
    instrument_name=instrument_name,
    from_resolution=from_resolution,
    solar_lines_csv=globals().get("solar_lines_csv"),
    atmosphere_layers=atmosphere_layers,
    solar_abundances=solar_abundances,
    teff=teff,
    logg=logg,
    mh=mh,
    alpha=alpha,
    vmic=vmic,
    code=code,
    output_abundances_result_path_linebyline_q2=output_abundances_result_path_linebyline_q2,
)
raw_path = _lbl["raw_path"]
print(_lbl["summary_df"].head(10))



In [ ]:
step_log("7", "Run HFS processing and update line-by-line outputs.")
RUN_HFS_BLENDS = True
RUN_HFS_BLENDS_DRY_RUN = False

_hfs = run_hfs_blends_and_update_outputs(
    ispec=ispec,
    target=target,
    output_folder=output_folder,
    ispec_dir=ispec_dir,
    atmosphere_layers=atmosphere_layers,
    teff=teff,
    logg=logg,
    mh=mh,
    vmic=vmic,
    solar_label=solar_label,
    output_abundances_result_path_linebyline_q2=output_abundances_result_path_linebyline_q2,
    pipeline_options=PIPELINE_OPTIONS,
    run_hfs_blends=RUN_HFS_BLENDS,
    dry_run=RUN_HFS_BLENDS_DRY_RUN,
)
if _hfs.get("ran"):
    step_log(
        "7",
        f"HFS done: rows={_hfs['hfs_rows']}, replaced={_hfs['replaced_n']}, failed={_hfs['failed_hfs_n']}",
    )
    if _hfs.get("report_path"):
        print(f"HFS detailed report saved: {_hfs['report_path']}")



In [ ]:
from element_uncertainties import finalize_element_outputs
_uncertainties = finalize_element_outputs(
    ispec=ispec, target=target, linemasks=linemasks, ispec_dir=ispec_dir,
    output_folder=output_folder, solar_label=solar_label,
    summary_path=output_abundances_result_path_linebyline_q2,
    params=dict(teff=teff, logg=logg, MH=mh, alpha=alpha, vmic=vmic),
    parameter_errors=_ctx4a["parameter_errors"], pipeline_options=PIPELINE_OPTIONS,
)
